[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C60_Edge_Deployment_Consistency_Course/02_tensorrt/02_tensorrt_pipeline.ipynb)

# 02 · TensorRT 构建与优化（迷你图 IR / 融合规则引擎 / 动态 shape profile / engine 矩阵）

目标：把 TensorRT 里最不透明的三件事——**层融合**、**动态 shape 的代价**、**engine 与硬件的绑定**——
用可运行的数值钉死。全程纯 numpy，不装 TensorRT，也不需要 GPU。

本 notebook 你会亲手实现：
1. **numpy 版 Conv / BN / ReLU**，并给出 **Conv+BN 融合的数值等价证明**（`np.allclose` 到 $10^{-12}$）
2. 一个**迷你图 IR**（Node / Graph / 拓扑执行）+ **融合规则引擎**（模式匹配 + use-def 安全性检查）
3. **什么阻止了融合**：多消费者、debug 输出——两种阻断各构造一个图，跑出来看
4. **roofline 代价模型**：融合前后的 FLOPs / 访存字节 / 算术强度 / 估计耗时
   → 得到关键结论：**融合对标准卷积只值 ~9%，对 depthwise 值 ~67%**
5. **动态 shape 的 optimization profile**：代价模型 + **动态规划求最优 k 档切分**
6. **构建时间 × 硬件矩阵**的组合爆炸计算，以及 engine 兼容性检查器

> 心智模型：**ONNX 说「模型是什么」，engine 说「这块芯片打算怎么做」。两者的节点根本不是一一对应的。**

## 1 · numpy 版算子与 Conv+BN 融合的数值等价

先把三个算子写出来。卷积用最朴素的七重循环展开（小尺寸够用，胜在一眼能看懂）。

融合公式（模块正文第 4 节）：

$$\tilde W_o = \frac{\gamma_o}{\sqrt{\sigma_o^2+\epsilon}}W_o,\qquad
  \tilde b_o = \frac{\gamma_o (b_o-\mu_o)}{\sqrt{\sigma_o^2+\epsilon}} + \beta_o$$

**这是恒等变换，不是近似。**

In [ ]:
import numpy as np, math, copy
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

def conv2d(x, w, b, stride=1, pad=1, groups=1):
    """x:(C,H,W)  w:(O, C//groups, kh, kw)  b:(O,)  ->  (O,Ho,Wo)"""
    C, H, W = x.shape
    O, Cg, kh, kw = w.shape
    assert C == Cg * groups and O % groups == 0
    xp = np.pad(x, ((0, 0), (pad, pad), (pad, pad)))
    Ho = (H + 2 * pad - kh) // stride + 1
    Wo = (W + 2 * pad - kw) // stride + 1
    out = np.zeros((O, Ho, Wo))
    ocpg = O // groups
    for o in range(O):
        cs = (o // ocpg) * Cg                      # 该输出通道所属 group 的输入通道起点
        acc = np.zeros((Ho, Wo))
        for c in range(Cg):
            for i in range(kh):
                for j in range(kw):
                    acc += w[o, c, i, j] * xp[cs + c,
                                              i:i + Ho * stride:stride,
                                              j:j + Wo * stride:stride]
        out[o] = acc + b[o]
    return out

def batchnorm(x, gamma, beta, mean, var, eps=1e-5):
    """推理期 BN：逐通道仿射，running stats 已冻结"""
    s = gamma / np.sqrt(var + eps)
    return x * s[:, None, None] + (beta - mean * s)[:, None, None]

def relu(x):
    return np.maximum(x, 0.0)

def fuse_conv_bn(w, b, gamma, beta, mean, var, eps=1e-5):
    """把推理期 BN 吸进上一层卷积的权重与偏置里。返回 (w_new, b_new)。"""
    s = gamma / np.sqrt(var + eps)
    return w * s[:, None, None, None], (b - mean) * s + beta

print('✅ 算子就位：conv2d / batchnorm / relu / fuse_conv_bn')

In [ ]:
# —— 数值等价验证：随机权重 + 随机输入 ——
C_IN, C_OUT, K, HW = 16, 16, 3, 16
x  = rng.standard_normal((C_IN, HW, HW))
W  = rng.standard_normal((C_OUT, C_IN, K, K)) * 0.15
B  = rng.standard_normal(C_OUT) * 0.1
gm = rng.uniform(0.5, 1.8, C_OUT)
bt = rng.standard_normal(C_OUT) * 0.3
mu = rng.standard_normal(C_OUT) * 0.4
va = rng.uniform(0.2, 2.5, C_OUT)

y_ref = batchnorm(conv2d(x, W, B), gm, bt, mu, va)          # Conv 然后 BN
W2, B2 = fuse_conv_bn(W, B, gm, bt, mu, va)
y_fus = conv2d(x, W2, B2)                                   # 融合成一个 Conv

err = np.abs(y_ref - y_fus).max()
print('两条路径的最大逐元素误差 =', err)
assert np.allclose(y_ref, y_fus, atol=1e-11), err
assert err < 1e-11

# 手算一个单通道例子，确认公式方向没搞反
w1 = np.array([[[[2.0]]]]); b1 = np.array([1.0])
g1 = np.array([3.0]); be1 = np.array([0.5]); m1 = np.array([4.0]); v1 = np.array([4.0 - 1e-5])
x1 = np.array([[[5.0]]])
# conv: 2*5+1 = 11 ; bn: 3*(11-4)/2 + 0.5 = 11.0
w1f, b1f = fuse_conv_bn(w1, b1, g1, be1, m1, v1)
print('手算：s=3/2=1.5, W~=2*1.5=%.4f, b~=(1-4)*1.5+0.5=%.4f' % (w1f[0,0,0,0], b1f[0]))
assert abs(w1f[0, 0, 0, 0] - 3.0) < 1e-6
assert abs(b1f[0] - (-4.0)) < 1e-6
assert abs(conv2d(x1, w1f, b1f, pad=0)[0, 0, 0] - 11.0) < 1e-6
print('✅ Conv+BN 融合是**严格恒等**（浮点舍入除外）——这就是 TensorRT 敢无条件做它的原因。')
print('   同一套代数还支撑：RepVGG 的重参数化（C53 m01）与 QAT 里的 BN folding（下一模块）。')

## 2 · 迷你图 IR：Node / Graph / 拓扑执行

一个能表达 ONNX 子集的最小 IR。关键在于 `Graph` 要显式记住 **outputs**——
后面判断「能不能融合」时，「这个张量是不是图的输出」是决定性条件之一。

In [ ]:
class Node:
    def __init__(self, name, op, inputs, outputs, attrs=None):
        self.name, self.op = name, op
        self.inputs, self.outputs = list(inputs), list(outputs)
        self.attrs = dict(attrs or {})
    def __repr__(self):
        return '<%s %s>' % (self.op, self.name)

class Graph:
    def __init__(self, nodes, inputs, outputs, inits=None):
        self.nodes = list(nodes)          # 假定已是拓扑序
        self.inputs, self.outputs = list(inputs), list(outputs)
        self.inits = dict(inits or {})    # 常量（权重）
    def clone(self):
        return copy.deepcopy(self)
    def op_count(self):
        d = {}
        for n in self.nodes:
            d[n.op] = d.get(n.op, 0) + 1
        return d

def run_graph(g, feed):
    env = dict(g.inits); env.update(feed)
    for n in g.nodes:
        xs = [env[t] for t in n.inputs]
        if n.op == 'Conv':
            y = conv2d(xs[0], xs[1], xs[2], n.attrs.get('stride', 1),
                       n.attrs.get('pad', 1), n.attrs.get('groups', 1))
            if n.attrs.get('act') == 'relu':          # 融合进来的激活
                y = relu(y)
        elif n.op == 'BatchNormalization':
            y = batchnorm(xs[0], xs[1], xs[2], xs[3], xs[4], n.attrs.get('eps', 1e-5))
        elif n.op == 'Relu':
            y = relu(xs[0])
        elif n.op == 'Add':
            y = xs[0] + xs[1]
        else:
            raise ValueError('unknown op ' + n.op)
        env[n.outputs[0]] = y
    return {t: env[t] for t in g.outputs}

def consumers(g, t):
    return [n for n in g.nodes if t in n.inputs]

print('✅ 迷你 IR 就位')

In [ ]:
def make_block(i, tin, tout, inits, c=16, k=3, groups=1):
    """造一个 Conv->BN->ReLU 三件套，返回 3 个 Node，并把权重写进 inits"""
    r = np.random.default_rng(100 + i)
    cg = c // groups
    inits['w%d' % i]  = r.standard_normal((c, cg, k, k)) * 0.12
    inits['b%d' % i]  = r.standard_normal(c) * 0.05
    inits['g%d' % i]  = r.uniform(0.6, 1.6, c)
    inits['be%d' % i] = r.standard_normal(c) * 0.2
    inits['m%d' % i]  = r.standard_normal(c) * 0.3
    inits['v%d' % i]  = r.uniform(0.3, 2.0, c)
    t1, t2 = 'c%d_out' % i, 'bn%d_out' % i
    return [
        Node('conv%d' % i, 'Conv', [tin, 'w%d' % i, 'b%d' % i], [t1],
             {'pad': k // 2, 'groups': groups}),
        Node('bn%d' % i, 'BatchNormalization',
             [t1, 'g%d' % i, 'be%d' % i, 'm%d' % i, 'v%d' % i], [t2]),
        Node('relu%d' % i, 'Relu', [t2], [tout]),
    ]

def make_chain(n_blocks=6, c=16):
    inits, nodes, t = {}, [], 'x'
    for i in range(n_blocks):
        tout = 'y%d' % i
        nodes += make_block(i, t, tout, inits, c=c)
        t = tout
    return Graph(nodes, ['x'], [t], inits)

G = make_chain(6)
X = {'x': rng.standard_normal((16, 16, 16))}
out_ref = run_graph(G, X)['y5']
print('链式图：%d 个节点  %s' % (len(G.nodes), G.op_count()))
print('输出形状', out_ref.shape, ' 均值 %.4f' % out_ref.mean())
assert len(G.nodes) == 18 and G.op_count() == {'Conv': 6, 'BatchNormalization': 6, 'Relu': 6}
print('✅ 一个 6 block 的骨干：ONNX 里是 18 个节点。engine 里应该是几个？下一节见分晓。')

## 3 · 融合规则引擎：模式匹配 + use-def 安全性检查

规则只有两条，但**安全性检查才是重点**：

- `Conv → BatchNormalization`（且 Conv 尚未吸收激活）→ 合并权重，BN 消失
- `Conv → Relu` → 把激活写进 Conv 的 `act` 属性，Relu 消失

**安全性条件（两条都要满足，缺一不可）**：中间张量的消费者数 == 1，且它不是图的输出。

In [ ]:
def fusible(g, t):
    """张量 t 能否被「吃掉」：唯一消费者 且 不是图输出"""
    return len(consumers(g, t)) == 1 and t not in g.outputs

def blocked_reason(g, t):
    if t not in [o for n in g.nodes for o in n.outputs]:
        return '不是中间张量'
    if t in g.outputs:
        return '是图的输出 -> 必须实体化'
    k = len(consumers(g, t))
    if k > 1:
        return '有 %d 个消费者 -> 必须实体化' % k
    if k == 0:
        return '没有消费者（死张量）'
    return ''

def fuse_graph(g):
    """反复应用融合规则直到不动点。返回 (新图, 融合日志)"""
    g, log = g.clone(), []
    changed, uid = True, 0
    while changed:
        changed = False
        for n in list(g.nodes):
            if n.op != 'Conv' or n.attrs.get('act') is not None:
                continue
            t = n.outputs[0]
            if not fusible(g, t):
                continue
            c = consumers(g, t)[0]
            if c.op == 'BatchNormalization':
                W, B = g.inits[n.inputs[1]], g.inits[n.inputs[2]]
                gm, bt, mu, va = [g.inits[k] for k in c.inputs[1:5]]
                W2, B2 = fuse_conv_bn(W, B, gm, bt, mu, va, c.attrs.get('eps', 1e-5))
                uid += 1
                nw, nb = 'w_f%d' % uid, 'b_f%d' % uid
                g.inits[nw], g.inits[nb] = W2, B2
                n.inputs[1], n.inputs[2] = nw, nb
                n.outputs[0] = c.outputs[0]
                n.name = n.name + ' + ' + c.name
                g.nodes.remove(c); log.append(('Conv+BN', n.name)); changed = True
            elif c.op == 'Relu':
                n.attrs['act'] = 'relu'
                n.outputs[0] = c.outputs[0]
                n.name = n.name + ' + ' + c.name
                g.nodes.remove(c); log.append(('Conv+Act', n.name)); changed = True
    return g, log

print('✅ 融合规则引擎就位（2 条规则 + 2 条安全性条件）')

In [ ]:
GF, log = fuse_graph(G)
out_fus = run_graph(GF, X)['y5']

print('融合前：%2d 个节点  %s' % (len(G.nodes), G.op_count()))
print('融合后：%2d 个节点  %s' % (len(GF.nodes), GF.op_count()))
print()
print('engine 里的层名（TensorRT 就是这样把被融的层名拼起来的）：')
for n in GF.nodes:
    print('   ', n.name)

err = np.abs(out_ref - out_fus).max()
print()
print('融合前后输出的最大误差 =', err)
assert np.allclose(out_ref, out_fus, atol=1e-11), err
assert len(GF.nodes) == 6 and GF.op_count() == {'Conv': 6}
assert len(log) == 12                       # 6 次 Conv+BN + 6 次 Conv+Act
print('✅ 18 个 ONNX 节点 -> 6 个 engine 层，数值完全一致。')
print('   **拿 ONNX 节点数去猜 engine 性能，方向就是错的。**')

## 4 · 什么阻止了融合：两种阻断，各造一个图

- **阻断 A：中间张量有两个消费者**（残差分支直接接在 BN 输出上）
- **阻断 B：中间张量被 mark 成图的输出**（为了 debug 多导出了一个特征图）

两种都**不会报错**，只会安静地少融一次。

In [ ]:
def make_residual_graph():
    """conv0 -> bn0 -> relu0 -> ... 但 bn0 的输出**同时**被 Add 用掉"""
    inits = {}
    nodes = make_block(0, 'x', 'r0', inits)          # conv0/bn0/relu0，relu0 输出 r0
    nodes.append(Node('add0', 'Add', ['r0', 'bn0_out'], ['y']))
    return Graph(nodes, ['x'], ['y'], inits)

def make_debug_graph():
    """完全正常的链，只是有人为了 debug 把 conv0 的输出也 mark 成了图输出"""
    inits = {}
    nodes = make_block(0, 'x', 'y', inits)
    return Graph(nodes, ['x'], ['y', 'c0_out'], inits)   # ← 多了一个输出

for name, g in [('① 正常链', make_chain(1)),
                ('② 残差接在 BN 输出上', make_residual_graph()),
                ('③ 多 mark 了一个 debug 输出', make_debug_graph())]:
    gf, _ = fuse_graph(g)
    print('%-24s 融合前 %2d 层 -> 融合后 %2d 层   %s'
          % (name, len(g.nodes), len(gf.nodes), gf.op_count()))
    for n in g.nodes:
        t = n.outputs[0]
        r = blocked_reason(g, t)
        print('        %-8s -> %-9s : %s' % (n.name, t, r or '可被下游吃掉 ✅'))
    print()

g1f, _ = fuse_graph(make_chain(1))
g2f, _ = fuse_graph(make_residual_graph())
g3f, _ = fuse_graph(make_debug_graph())
assert len(g1f.nodes) == 1                       # Conv
assert g2f.op_count().get('BatchNormalization', 0) == 0 and g2f.op_count()['Relu'] == 1
assert g3f.op_count()['BatchNormalization'] == 1  # Conv+BN 被 debug 输出挡住了
print('结论：')
print('  ② BN 的输出有 2 个消费者 -> Conv+BN 仍能融，但 **ReLU 融不进去**')
print('  ③ Conv 的输出被 mark 成图输出 -> **Conv+BN 直接融不了**')
print('⚠️  两种情况数值都完全正确、都不报错。**只有延迟会告诉你出事了。**')

## 5 · roofline 代价模型：融合到底省了多少

$$t_{\text{layer}} \approx \max\!\left(\frac{\text{FLOPs}}{P_{\text{peak}}},\ \frac{\text{Bytes}}{B_{\text{mem}}}\right) + t_{\text{launch}}$$

用一个典型边缘 SoC 的参数：**10 TFLOPS FP16 + 200 GB/s 带宽 + 5 µs kernel launch**。

In [ ]:
PEAK   = 10e12      # FLOP/s (FP16)
BW     = 200e9      # B/s
LAUNCH = 5e-6       # s，每个 kernel 的启动开销
DB     = 2          # FP16 每元素字节数

def roofline_us(flops, byts, peak=PEAK, bw=BW, launch=LAUNCH):
    return (max(flops / peak, byts / bw) + launch) * 1e6

def conv_cost(H, W, cin, cout, k, groups=1):
    cg = cin // groups
    flops = 2 * H * W * cout * cg * k * k
    byts  = (H * W * cin + H * W * cout + cout * cg * k * k) * DB
    return flops, byts

def elem_cost(H, W, c, ops_per_elem):
    """逐元素算子：算 ops_per_elem 次，读一遍写一遍"""
    return float(H * W * c * ops_per_elem), float(2 * H * W * c * DB)

H = W = 80; C = 256
cases = {
    '3x3 标准卷积 256->256': conv_cost(H, W, C, C, 3, groups=1),
    '5x5 depthwise  256':    conv_cost(H, W, C, C, 5, groups=C),
}
bn_f, bn_b   = elem_cost(H, W, C, 2)
rl_f, rl_b   = elem_cost(H, W, C, 1)

print('%-24s %10s %10s %10s %12s %12s %8s' %
      ('层', 'GFLOPs', 'MB(拆开)', 'MB(融合)', 'AI(拆开)', 'AI(融合)', '省%'))
saving = {}
for name, (f, b) in cases.items():
    f_un, b_un = f + bn_f + rl_f, b + bn_b + rl_b
    t_un = roofline_us(f, b) + roofline_us(bn_f, bn_b) + roofline_us(rl_f, rl_b)
    t_fu = roofline_us(f_un, b)                       # 融合后：一趟访存，算力照旧
    saving[name] = (t_un, t_fu, 1 - t_fu / t_un)
    print('%-24s %10.3f %10.2f %10.2f %12.1f %12.1f %7.0f%%' %
          (name, f / 1e9, b_un / 1e6, b / 1e6, f_un / b_un, f_un / b, 100 * (1 - t_fu / t_un)))

print()
print('%-24s %14s %14s' % ('层', '拆成 3 个 kernel', '融合成 1 个'))
for name, (t_un, t_fu, s) in saving.items():
    print('%-24s %12.1f µs %12.1f µs' % (name, t_un, t_fu))

s_std = saving['3x3 标准卷积 256->256'][2]
s_dw  = saving['5x5 depthwise  256'][2]
assert 0.05 < s_std < 0.15, s_std
assert 0.60 < s_dw  < 0.75, s_dw
print()
print('⚠️  **同一个融合，对标准卷积值 %.0f%%，对 depthwise 值 %.0f%%。**' % (100 * s_std, 100 * s_dw))
print('   因为 depthwise 本来就是 memory-bound（算术强度只有 %.1f）——'
      % (cases['5x5 depthwise  256'][0] / cases['5x5 depthwise  256'][1]))
print('   它不缺算力，缺带宽，而融合省的正是带宽。')
print('   推论：**越是为「FLOPs 少」设计的轻量模型，越依赖融合，也越怕融合被阻断。**')

In [ ]:
# 把代价模型接到图上：估计整张图的延迟，并量化两种阻断的代价
def graph_latency_us(g, H=80, W=80, C=256, k=3, groups=1):
    ef, eb = elem_cost(H, W, C, 2)          # BN
    af, ab = elem_cost(H, W, C, 1)          # ReLU
    tot = 0.0
    for n in g.nodes:
        if n.op == 'Conv':
            f, b = conv_cost(H, W, C, C, k, groups)
            if n.attrs.get('act') == 'relu':
                f += af                      # 激活融进 epilogue：多算，但不多访存
            tot += roofline_us(f, b)
        elif n.op == 'BatchNormalization':
            tot += roofline_us(ef, eb)
        elif n.op == 'Relu':
            tot += roofline_us(af, ab)
        elif n.op == 'Add':
            tot += roofline_us(af, ab * 1.5)
    return tot

# 事故重演：为了排查一个 badcase，把第 3 个 block 的 conv 输出也 mark 成了图输出
G_dbg = G.clone(); G_dbg.outputs = G_dbg.outputs + ['c3_out']
G_fus, _ = fuse_graph(G)
G_dbf, _ = fuse_graph(G_dbg)
print('层数：原图 %d  ->  正常融合 %d  ->  多一个 debug 输出 %d'
      % (len(G.nodes), len(G_fus.nodes), len(G_dbf.nodes)))
assert len(G_fus.nodes) == 6 and len(G_dbf.nodes) == 8

BACKBONES = [('标准卷积骨干 (3x3, 256->256)', dict(k=3, groups=1)),
             ('depthwise 骨干 (5x5, dw)',      dict(k=5, groups=256))]
print()
print('%-30s %12s %12s %12s %10s' %
      ('骨干类型', '不融合 µs', '正常融合 µs', 'debug输出 µs', '事故代价'))
costs_dbg = {}
for name, kw in BACKBONES:
    t_full = graph_latency_us(G, **kw)
    t_fuse = graph_latency_us(G_fus, **kw)
    t_dbg  = graph_latency_us(G_dbf, **kw)
    costs_dbg[name] = (t_full, t_fuse, t_dbg)
    print('%-30s %12.1f %12.1f %12.1f %9.1f%%'
          % (name, t_full, t_fuse, t_dbg, 100 * (t_dbg / t_fuse - 1)))

(_, f_std, d_std) = costs_dbg['标准卷积骨干 (3x3, 256->256)']
(_, f_dw,  d_dw)  = costs_dbg['depthwise 骨干 (5x5, dw)']
assert d_std > f_std and d_dw > f_dw
assert (d_std / f_std - 1) < 0.05, '标准卷积骨干上，这个事故只值几个百分点'
assert (d_dw / f_dw - 1) > 0.25, 'depthwise 骨干上，同一个事故的代价大一个数量级'
print()
print('⚠️  **同一行代码，在标准卷积骨干上只涨 %.1f%%，在 depthwise 骨干上涨 %.0f%%。**'
      % (100 * (d_std / f_std - 1), 100 * (d_dw / f_dw - 1)))
print('   而现代实时检测器（RTMDet 的 CSPNeXt、轻量 YOLO）恰恰是后者。')
print('   这类事故的共同特征：**数值全对、评测全过、只有延迟涨了**。')
print('   对策：把 engine 层数 / 融合层占比 / 输出张量数做成 CI 门禁 ——')
print('   一致性检查抓不到它，因为数值本来就是对的。')

## 6 · 动态 shape：optimization profile 的代价模型与最优切分

代价模型（模块正文第 3 节）：

$$\mathbb{E}[T]=\sum_s p(s)\,t_0(s)\Bigl(1+\alpha\bigl|\ln (s/s_{\text{opt}})\bigr|\Bigr)$$

场景取自**两级 TSR 架构的第二级**：检测器给出的 crop 数随场景剧烈变化，
高速上常常只有 1–2 块标志，城市路口龙门架一次能出十几块。

In [ ]:
ALPHA = 0.15          # 偏离 opt 的惩罚系数（GEMM 类层实测常在 0.05~0.25）

def t0(s):
    """batch=s 时，为它专门构建的静态 engine 的耗时（ms）：固定开销 + 线性项"""
    return 0.80 + 0.25 * s

def t_with_opt(s, s_opt, alpha=ALPHA):
    return t0(s) * (1.0 + alpha * abs(math.log(s / s_opt)))

# 线上 batch 分布：双峰（高速稀疏场景 + 城市密集场景）
SHAPES = list(range(1, 25))
w = np.array([np.exp(-((s - 2) ** 2) / 4.0) * 3.0 + np.exp(-((s - 16) ** 2) / 18.0)
              for s in SHAPES])
PROBS = (w / w.sum()).tolist()

print('batch 分布（截断显示）：')
for s, p in zip(SHAPES, PROBS):
    if p > 0.008:
        print('  batch=%2d  p=%.3f  %s' % (s, p, '#' * int(p * 200)))
assert abs(sum(PROBS) - 1) < 1e-12

def expected_latency(shapes, probs, s_opt, alpha=ALPHA):
    return sum(p * t_with_opt(s, s_opt, alpha) for s, p in zip(shapes, probs))

print()
print('%-28s %12s' % ('单 profile 的 opt 取值', '期望延迟 ms'))
best = min(SHAPES, key=lambda o: expected_latency(SHAPES, PROBS, o))
for o in [1, 2, 4, 8, 12, 16, 24]:
    tag = '  <- 最优' if o == best else ''
    print('%-28s %12.4f%s' % ('opt = %d' % o, expected_latency(SHAPES, PROBS, o), tag))
print('全局最优 opt =', best, ' 期望延迟 %.4f ms' % expected_latency(SHAPES, PROBS, best))
ideal = sum(p * t0(s) for s, p in zip(SHAPES, PROBS))
print('理想（每个 shape 都有专属静态 engine）= %.4f ms' % ideal)
print('单 profile 的代价 = +%.1f%%' % (100 * (expected_latency(SHAPES, PROBS, best) / ideal - 1)))
assert expected_latency(SHAPES, PROBS, best) > ideal

In [ ]:
def best_partition(shapes, probs, k, alpha=ALPHA):
    """动态规划：把有序 shape 列表切成 k 段连续区间，每段一个 profile，
       使期望延迟最小。返回 (最小期望延迟, [(区间, opt), ...])。"""
    n = len(shapes)
    seg = [[(math.inf, None)] * (n + 1) for _ in range(n)]     # seg[i][j] = 区间 [i,j)
    for i in range(n):
        for j in range(i + 1, n + 1):
            bo, bc = None, math.inf
            for o in range(i, j):
                c = sum(probs[t] * t_with_opt(shapes[t], shapes[o], alpha) for t in range(i, j))
                if c < bc:
                    bc, bo = c, shapes[o]
            seg[i][j] = (bc, bo)
    INF = math.inf
    dp = [[INF] * (n + 1) for _ in range(k + 1)]
    bk = [[None] * (n + 1) for _ in range(k + 1)]
    dp[0][0] = 0.0
    for q in range(1, k + 1):
        for j in range(1, n + 1):
            for i in range(q - 1, j):
                if dp[q - 1][i] + seg[i][j][0] < dp[q][j]:
                    dp[q][j] = dp[q - 1][i] + seg[i][j][0]; bk[q][j] = i
    parts, j = [], n
    for q in range(k, 0, -1):
        i = bk[q][j]
        parts.append(((shapes[i], shapes[j - 1]), seg[i][j][1]))
        j = i
    return dp[k][n], parts[::-1]

print('%4s %14s %10s %s' % ('k', '期望延迟 ms', '相对理想', 'profile 切分 (min~max : opt)'))
prev = math.inf
costs = []
for k in range(1, 6):
    c, parts = best_partition(SHAPES, PROBS, k)
    costs.append(c)
    desc = '  '.join('[%d~%d:%d]' % (a, b, o) for (a, b), o in parts)
    print('%4d %14.4f %9.2f%% %s' % (k, c, 100 * (c / ideal - 1), desc))
    assert c <= prev + 1e-12, '增加 profile 不可能变差'
    prev = c
costs = np.array(costs)
gain = (costs[0] - costs) / (costs[0] - ideal)
print()
print('边际收益（相对「单 profile → 理想」这段差距）：', ' '.join('%.0f%%' % (100 * g) for g in gain))
assert costs[1] < costs[0]
assert gain[1] > 0.4, '第 2 个 profile 应吃掉大部分收益'
assert costs[4] - costs[3] > -0.02 * costs[0], '第 5 个 profile 已几乎无收益'
print('✅ **k=2 吃掉大部分收益，k>=4 基本是白花构建时间。**')

In [ ]:
# 另一半账：显存。**每个 profile 需要一个 execution context，各自按自己的 max 预留 activation**
M_WEIGHTS = 90.0                 # MB，权重（所有 context 共享）
M_ACT_PER_UNIT = 8.0             # MB / batch unit，激活显存近似线性于 batch

def memory_mb(parts):
    return M_WEIGHTS + sum(M_ACT_PER_UNIT * b for (_, b), _ in parts), len(parts)

print('%4s %13s %11s %9s %12s' % ('k', '期望延迟 ms', '显存 MB', 'context', '延迟↓ / 显存↑'))
m1 = None
for k in range(1, 5):
    c, parts = best_partition(SHAPES, PROBS, k)
    m, n_ctx = memory_mb(parts)
    if m1 is None:
        c1_, m1 = c, m
    print('%4d %13.4f %11.0f %9d %11s'
          % (k, c, m, n_ctx, '%.1f%% / +%.0f%%' % (100 * (1 - c / c1_), 100 * (m / m1 - 1))))
m_k1 = memory_mb(best_partition(SHAPES, PROBS, 1)[1])[0]
m_k4 = memory_mb(best_partition(SHAPES, PROBS, 4)[1])[0]
assert m_k4 > m_k1, '多 profile 必然多占显存'

# 分桶（bucketing）：把 batch padding 到固定档位，**每档一个静态 shape engine**
def bucket_eval(buckets):
    bo = lambda s: min(b for b in buckets if b >= s)
    t = sum(p * t0(bo(s)) for s, p in zip(SHAPES, PROBS))
    w = sum(p * (bo(s) - s) for s, p in zip(SHAPES, PROBS))
    return t, w

c1, _ = best_partition(SHAPES, PROBS, 1)
print()
print('%-34s %12s %14s %8s' % ('分桶方案（每档一个静态 engine）', '期望延迟 ms', '平均 padding 浪费', 'engine 数'))
for buckets in ([2, 4, 8, 16, 24], [1, 2, 3, 4, 6, 8, 12, 16, 20, 24]):
    t, w = bucket_eval(buckets)
    print('%-34s %12.4f %13.2f 个 %8d' % (str(buckets), t, w, len(buckets)))
t_coarse, _ = bucket_eval([2, 4, 8, 16, 24])
t_fine, _   = bucket_eval([1, 2, 3, 4, 6, 8, 12, 16, 20, 24])
print()
print('单 profile 动态 shape       %.4f ms' % c1)
print('理想（每 shape 专属 engine）%.4f ms' % ideal)
assert t_coarse > c1, '粗分桶：padding 浪费的算力超过了 profile 的偏离惩罚'
assert t_fine < c1,  '细分桶：静态 tactic 的收益压过了 padding 浪费'
print()
print('⚠️  **分桶不是无脑更优——桶的粒度决定成败。**')
print('   粗桶 %s：静态 tactic 很快，但平均要多算 1.89 个样本 -> 反而比单 profile 慢 %.1f%%'
      % ([2, 4, 8, 16, 24], 100 * (t_coarse / c1 - 1)))
print('   细桶（10 档）：padding 浪费降到 0.69 个 -> 比单 profile 快 %.1f%%，代价是 10 个 engine 要管'
      % (100 * (1 - t_fine / c1)))
print()
print('三条路没有普适赢家：')
print('   · 单 profile —— 最省事，主峰之外慢 10~30%')
print('   · 多 profile —— 更快，但**显存 × context 数**，车规 SoC 上这常常才是硬约束')
print('   · 分桶静态   —— 上限最高，但要同时付「padding 算力」和「多 engine 管理」')
print('   TSR 落点：主检测器一律**静态 shape**；只有两级架构的第二级分类器需要算这套账。')

## 7 · engine 矩阵与硬件绑定：组合爆炸算一遍

In [ ]:
def engine_matrix(platforms, precisions, resolutions, model_versions=1, trt_versions=1):
    n = (len(platforms) * len(precisions) * len(resolutions)
         * model_versions * trt_versions)
    return n

PLATFORMS   = ['Orin-N', 'Orin-X', 'DualOrin', 'Thor']
PRECISIONS  = ['fp16', 'int8']
RESOLUTIONS = ['960x540', '1280x720', '1920x1080']
MODEL_VERS  = 3          # 灰度期同时在网的模型版本
BUILD_MIN   = 12.0
SIZE_MB     = 80
MACHINES    = 6

n_per_ver = engine_matrix(PLATFORMS, PRECISIONS, RESOLUTIONS)
n_total   = engine_matrix(PLATFORMS, PRECISIONS, RESOLUTIONS, MODEL_VERS)
serial_h  = n_total * BUILD_MIN / 60
wall_h    = serial_h / MACHINES
store_gb  = n_total * SIZE_MB / 1024

print('平台 %d × 精度 %d × 分辨率 %d          = %d 个 engine / 每个模型版本'
      % (len(PLATFORMS), len(PRECISIONS), len(RESOLUTIONS), n_per_ver))
print('× 在网模型版本 %d                        = %d 个 engine' % (MODEL_VERS, n_total))
print('× 构建 %.0f 分钟                          = %.1f 机器小时（串行）' % (BUILD_MIN, serial_h))
print('÷ %d 台并行构建机                        = %.1f 小时墙钟' % (MACHINES, wall_h))
print('存储与分发                               = %.1f GB' % store_gb)
print('**独立验证次数（每个 engine 都是一个发布物）= %d 次完整评测**' % n_total)
assert n_per_ver == 24 and n_total == 72
assert abs(serial_h - 14.4) < 1e-9

print()
print('TensorRT 一次小版本升级会发生什么：')
print('  · 序列化格式变了 -> 所有 engine 作废')
print('  · **全量重建 %d 个 + 全量回归 %d 次** -> %.1f 小时墙钟 + 全部评测预算'
      % (n_total, n_total, wall_h))
print('  · 这就是为什么量产系统的 TRT 版本升级频率极低（常常一年一次或更少）')

# 用 hardware/version compatible 模式压缩矩阵：拿性能换维护性
LOSS = 0.10          # 典型 5~15%
n_compat = engine_matrix(['AmperePlus'], PRECISIONS, RESOLUTIONS, MODEL_VERS)
print()
print('如果启用 hardware-compatible（只用架构通用 tactic）：')
print('  engine 数 %d -> %d（少 %.0f%%），但**每个都慢约 %.0f%%**'
      % (n_total, n_compat, 100 * (1 - n_compat / n_total), 100 * LOSS))
budget_ms, base_ms = 12.0, 11.0
print('  延迟预算 %.1f ms、当前 %.1f ms：加 %.0f%% 后 = %.2f ms -> %s'
      % (budget_ms, base_ms, 100 * LOSS, base_ms * (1 + LOSS),
         '仍在预算内，划算' if base_ms * (1 + LOSS) <= budget_ms else '**超预算，不能换**'))
assert n_compat == 18
assert base_ms * (1 + LOSS) > budget_ms
print('  -> 本例中余量不够，只能老老实实维护 %d 个 engine。' % n_total)

In [ ]:
def engine_compatible(engine_meta, runtime_meta):
    """返回 (能否加载, 原因)。顺序很重要：先查会**硬失败**的，再查会**静默变慢**的。"""
    if engine_meta['trt'] != runtime_meta['trt'] and not engine_meta.get('version_compatible'):
        return False, 'TensorRT 版本不匹配 (%s vs %s) -> 反序列化失败' % (
            engine_meta['trt'], runtime_meta['trt'])
    if engine_meta['sm'] != runtime_meta['sm'] and not engine_meta.get('hw_compatible'):
        return False, 'SM 架构不匹配 (%s vs %s) -> 反序列化失败' % (
            engine_meta['sm'], runtime_meta['sm'])
    if runtime_meta['driver'] < engine_meta['min_driver']:
        return False, '驱动版本过低 (%.1f < %.1f)' % (
            runtime_meta['driver'], engine_meta['min_driver'])
    if engine_meta['gpu'] != runtime_meta['gpu']:
        return True, '⚠️ 能加载，但 tactic 是为 %s 挑的，在 %s 上**不是最优**（静默变慢）' % (
            engine_meta['gpu'], runtime_meta['gpu'])
    return True, 'OK'

ENG = dict(trt='8.6.1', sm='sm_87', gpu='Orin-X', min_driver=35.0)
CASES = [
    ('同型号同版本',        dict(trt='8.6.1', sm='sm_87', gpu='Orin-X',  driver=35.4)),
    ('TRT 小版本不同',      dict(trt='8.6.2', sm='sm_87', gpu='Orin-X',  driver=35.4)),
    ('换到 Thor（新架构）', dict(trt='8.6.1', sm='sm_90', gpu='Thor',    driver=36.0)),
    ('同架构不同型号',      dict(trt='8.6.1', sm='sm_87', gpu='Orin-N',  driver=35.4)),
    ('驱动过旧',            dict(trt='8.6.1', sm='sm_87', gpu='Orin-X',  driver=34.1)),
]
for name, rt in CASES:
    ok, why = engine_compatible(ENG, rt)
    print('%-22s %-6s %s' % (name, '✅' if ok else '❌', why))

assert engine_compatible(ENG, CASES[0][1])[0]
assert not engine_compatible(ENG, CASES[1][1])[0]
assert not engine_compatible(ENG, CASES[2][1])[0]
ok, why = engine_compatible(ENG, CASES[3][1])
assert ok and '不是最优' in why           # ← 最危险的一档：不报错，只变慢
assert not engine_compatible(ENG, CASES[4][1])[0]
print()
print('⚠️  **最危险的是第 4 行**：能加载、精度对、只是慢。')
print('   前三种失败是显式的，集成阶段就会被抓到；第 4 种要靠「按硬件指纹选包」的分发机制来防。')

## ✏️ 练习 1：BN 折叠

实现 `fold_bn(w, b, gamma, beta, mean, var, eps=1e-5)`，返回融合后的 `(w_new, b_new)`：

$$\tilde W_o = \frac{\gamma_o}{\sqrt{\sigma_o^2+\epsilon}}W_o,\qquad
  \tilde b_o = \frac{\gamma_o (b_o-\mu_o)}{\sqrt{\sigma_o^2+\epsilon}} + \beta_o$$

注意 `w` 的形状是 `(O, C//groups, kh, kw)`，缩放要作用在**第 0 维（输出通道）**上。

In [ ]:
def fold_bn(w, b, gamma, beta, mean, var, eps=1e-5):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# (a) 手算：单通道 1x1 卷积  w=2 b=1 ; gamma=3 beta=0.5 mean=4 var=4-eps
w1 = np.array([[[[2.0]]]]); b1 = np.array([1.0])
g1 = np.array([3.0]); be1 = np.array([0.5]); m1 = np.array([4.0]); v1 = np.array([4.0 - 1e-5])
wf, bf = fold_bn(w1, b1, g1, be1, m1, v1)
assert abs(wf[0, 0, 0, 0] - 3.0) < 1e-6, wf          # 2 * (3/2)
assert abs(bf[0] - (-4.0)) < 1e-6, bf                # (1-4)*1.5 + 0.5
# (b) 随机张量上的严格等价
for gp in (1, 4, 16):
    r = np.random.default_rng(7 + gp)
    xx = r.standard_normal((16, 12, 12))
    ww = r.standard_normal((16, 16 // gp, 3, 3)) * 0.2
    bb = r.standard_normal(16) * 0.1
    gg = r.uniform(0.4, 2.0, 16); be = r.standard_normal(16)
    mm = r.standard_normal(16); vv = r.uniform(0.2, 3.0, 16)
    ref = batchnorm(conv2d(xx, ww, bb, groups=gp), gg, be, mm, vv)
    w2, b2 = fold_bn(ww, bb, gg, be, mm, vv)
    got = conv2d(xx, w2, b2, groups=gp)
    assert np.allclose(ref, got, atol=1e-11), (gp, np.abs(ref - got).max())
    print('groups=%2d  最大误差 %.2e  ✅' % (gp, np.abs(ref - got).max()))
print('✅ 练习 1 通过：BN 折叠对普通卷积与分组/depthwise 卷积同样成立。')

## ✏️ 练习 2：谁挡住了融合

实现两个函数：

- `absorbable(g, t)` → 张量 `t` 能否被下游「吃掉」：**唯一消费者 且 不是图的输出**
- `blocked_convs(g)` → 返回所有「输出无法被吃掉」的 `Conv` 节点名（排序后的 list）

这两个函数就是 TensorRT 融合安全性检查的最小版本。

In [ ]:
def absorbable(g, t):
    # TODO
    raise NotImplementedError

def blocked_convs(g):
    # TODO: 返回 sorted 的节点名列表
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
g_chain = make_chain(2)
g_res   = make_residual_graph()
g_dbg   = make_debug_graph()

assert absorbable(g_chain, 'c0_out') is True
assert absorbable(g_chain, 'bn0_out') is True
assert absorbable(g_chain, 'y1') is False, '图的输出不能被吃掉'
assert absorbable(g_res, 'bn0_out') is False, 'bn0_out 有 2 个消费者'
assert absorbable(g_dbg, 'c0_out') is False, 'c0_out 被 mark 成了图输出'

assert blocked_convs(g_chain) == [], blocked_convs(g_chain)
assert blocked_convs(g_res) == [], blocked_convs(g_res)      # conv0 的输出只有 bn0 用
assert blocked_convs(g_dbg) == ['conv0'], blocked_convs(g_dbg)

g_mix = make_chain(6); g_mix.outputs = g_mix.outputs + ['c1_out', 'c4_out']
assert blocked_convs(g_mix) == ['conv1', 'conv4'], blocked_convs(g_mix)
print('正常链         被挡住的 conv:', blocked_convs(g_chain))
print('残差图         被挡住的 conv:', blocked_convs(g_res), ' (挡住的是 ReLU，不是 BN)')
print('debug 输出图   被挡住的 conv:', blocked_convs(g_dbg))
print('多 debug 输出  被挡住的 conv:', blocked_convs(g_mix))
print('✅ 练习 2 通过：把这两行逻辑接进 CI，就能在 PR 阶段拦住「多 mark 一个输出」的事故。')

## ✏️ 练习 3：roofline 与融合收益

实现：

- `roofline_us2(flops, byts, peak=1e13, bw=2e11, launch=5e-6)` → 微秒
- `cbr_saving(H, W, c, k, groups=1)` → `(拆成三个 kernel 的 µs, 融合成一个的 µs, 节省比例)`

约定（与正文一致）：FP16 每元素 2 字节；卷积 FLOPs = `2*H*W*cout*(cin/groups)*k*k`；
BN 每元素 2 次运算、ReLU 1 次，两者都是「读一遍写一遍」；融合后**算力照旧、访存只剩卷积那一趟**。

In [ ]:
def roofline_us2(flops, byts, peak=1e13, bw=2e11, launch=5e-6):
    # TODO
    raise NotImplementedError

def cbr_saving(H, W, c, k, groups=1):
    # TODO: 返回 (t_unfused_us, t_fused_us, saving_frac)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# (a) 手算 roofline：compute-bound 与 memory-bound 各一例
assert abs(roofline_us2(1e12, 1e6) - 100005.0) < 1e-6, roofline_us2(1e12, 1e6)   # 0.1s + 5µs
assert abs(roofline_us2(1e6, 1e9) - 5005.0) < 1e-6, roofline_us2(1e6, 1e9)       # 5ms + 5µs
# (b) 正文的两个案例
t_un, t_fu, s_std = cbr_saving(80, 80, 256, 3, 1)
print('3x3 标准卷积 : 拆开 %.1f µs -> 融合 %.1f µs   省 %.1f%%' % (t_un, t_fu, 100 * s_std))
assert 0.08 < s_std < 0.10, s_std
t_un2, t_fu2, s_dw = cbr_saving(80, 80, 256, 5, 256)
print('5x5 depthwise: 拆开 %.1f µs -> 融合 %.1f µs   省 %.1f%%' % (t_un2, t_fu2, 100 * s_dw))
assert 0.65 < s_dw < 0.69, s_dw
assert s_dw > 6 * s_std, 'depthwise 的融合收益应远大于标准卷积'
# (c) 分辨率越高、通道越少，越 memory-bound，融合越值钱
_, _, s_p2 = cbr_saving(160, 160, 64, 3, 1)      # P2 层级：高分辨率 + 少通道
print('P2 层级 3x3 64ch (160x160): 省 %.1f%%' % (100 * s_p2))
assert s_p2 > s_std
print('✅ 练习 3 通过：**融合的价值 = 该层有多 memory-bound**，与它的 FLOPs 无关。')

## ✏️ 练习 4：optimization profile 的 opt 该选在哪

实现：

- `profile_cost(shapes, probs, s_opt, alpha=0.15)` → 期望延迟
  （用已定义的 `t0(s)` 与惩罚 $1+\alpha|\ln(s/s_{opt})|$）
- `pick_opt(shapes, probs, alpha=0.15)` → 在 `shapes` 里挑期望延迟最小的 `s_opt`

In [ ]:
def profile_cost(shapes, probs, s_opt, alpha=0.15):
    # TODO
    raise NotImplementedError

def pick_opt(shapes, probs, alpha=0.15):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# (a) opt 恰好等于唯一的 shape 时，没有惩罚
assert abs(profile_cost([4], [1.0], 4) - t0(4)) < 1e-12
assert abs(profile_cost([4], [1.0], 4) - 1.8) < 1e-12
# (b) 两点分布：频次一样，但**耗时大的那档应该拿走 opt**
c_lo = profile_cost([2, 8], [0.5, 0.5], 2)
c_hi = profile_cost([2, 8], [0.5, 0.5], 8)
print('shapes=[2,8] 各 50%%:  opt=2 -> %.4f ms   opt=8 -> %.4f ms' % (c_lo, c_hi))
assert c_hi < c_lo, 'opt 应偏向耗时大的一档，而不是频次中位数'
assert pick_opt([2, 8], [0.5, 0.5]) == 8
# (c) 复现第 6 节的结论
assert pick_opt(SHAPES, PROBS) == 15, pick_opt(SHAPES, PROBS)
assert abs(profile_cost(SHAPES, PROBS, 15) - 3.1924) < 1e-3
# (d) alpha=0 时，惩罚消失，任何 opt 都一样
assert abs(profile_cost(SHAPES, PROBS, 1, alpha=0.0)
           - profile_cost(SHAPES, PROBS, 24, alpha=0.0)) < 1e-12
print('全局最优 opt =', pick_opt(SHAPES, PROBS), '  期望延迟 %.4f ms'
      % profile_cost(SHAPES, PROBS, pick_opt(SHAPES, PROBS)))
print('✅ 练习 4 通过：**opt 不是「最常见的 shape」，是「加权后代价最小的 shape」**——')
print('   一个只出现 5% 但耗时是别人 10 倍的大 shape，完全可能把 opt 拉过去。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def fold_bn(w, b, gamma, beta, mean, var, eps=1e-5):
    s = gamma / np.sqrt(var + eps)
    return w * s[:, None, None, None], (b - mean) * s + beta

In [ ]:
# 练习 2 参考答案
def absorbable(g, t):
    return len([n for n in g.nodes if t in n.inputs]) == 1 and t not in g.outputs

def blocked_convs(g):
    return sorted(n.name for n in g.nodes
                  if n.op == 'Conv' and not absorbable(g, n.outputs[0]))

In [ ]:
# 练习 3 参考答案
def roofline_us2(flops, byts, peak=1e13, bw=2e11, launch=5e-6):
    return (max(flops / peak, byts / bw) + launch) * 1e6

def cbr_saving(H, W, c, k, groups=1):
    DBY = 2
    cg = c // groups
    cf = 2 * H * W * c * cg * k * k
    cb = (H * W * c + H * W * c + c * cg * k * k) * DBY
    bf, bb = H * W * c * 2, 2 * H * W * c * DBY      # BN
    af, ab = H * W * c * 1, 2 * H * W * c * DBY      # ReLU
    t_un = roofline_us2(cf, cb) + roofline_us2(bf, bb) + roofline_us2(af, ab)
    t_fu = roofline_us2(cf + af, cb)                 # 算力照旧，访存只剩卷积那一趟
    return t_un, t_fu, 1 - t_fu / t_un

In [ ]:
# 练习 4 参考答案
def profile_cost(shapes, probs, s_opt, alpha=0.15):
    return sum(p * t0(s) * (1 + alpha * abs(math.log(s / s_opt)))
               for s, p in zip(shapes, probs))

def pick_opt(shapes, probs, alpha=0.15):
    return min(shapes, key=lambda o: profile_cost(shapes, probs, o, alpha))

---
## 🧪 真实工程胶囊：TensorRT 构建脚本 + 必须进 CI 的三条门禁

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 构建：一条可以直接抄的 trtexec 命令（静态 shape 优先！）
# ══════════════════════════════════════════════════════════════════════
trtexec --onnx=det.onnx --saveEngine=det_orinx_fp16.plan \
        --fp16 \
        --memPoolSize=workspace:2048 \
        --timingCacheFile=golden_orinx_trt861.cache \
        --avgTiming=8 \
        --builderOptimizationLevel=3 \
        --profilingVerbosity=detailed \
        --exportLayerInfo=layers.json \
        --verbose 2>&1 | tee build.log

# 动态 shape 时三档都要给（**每个动态输入都要给**，漏一个直接报错）：
#   --minShapes=images:1x3x640x640
#   --optShapes=images:1x3x640x640      <- 这个数必须来自**线上 shape 分布统计**
#   --maxShapes=images:4x3x640x640      <- 显存按它预留，别随手写大

# 混合精度（下一模块会用到）：必须用 obey，不满足要它当场失败
#   --precisionConstraints=obey --layerPrecisions=head.reg.*:fp16

# ══════════════════════════════════════════════════════════════════════
# B. 三条必须进 CI 的门禁（一致性检查抓不到它们，因为数值本来就是对的）
# ══════════════════════════════════════════════════════════════════════
# 门禁 1 · 融合率：engine 层数 应显著少于 ONNX 节点数
#   python - <<PY
#   import json; L=json.load(open('layers.json'))['Layers']
#   fused = sum(1 for l in L if '+' in l['Name'])
#   print('layers=%d fused=%d ratio=%.2f' % (len(L), fused, fused/len(L)))
#   assert fused/len(L) > 0.5        # 阈值按你的模型基线定，重点是**不许回退**
#   PY
#
# 门禁 2 · 精度落实率：目标 FP16 时，FP32 层数必须为 0（或在白名单内）
#   grep -o '"Precision": *"[^"]*"' layers.json | sort | uniq -c
#   # 期望：FP16 绝大多数；出现大量 FP32 = 精度标志没生效
#
# 门禁 3 · Reformat 占比：精度/布局反复切换会把收益吃光
#   trtexec --loadEngine=det.plan --dumpProfile --separateProfileRun \
#           --exportProfile=prof.json --shapes=images:1x3x640x640
#   # 统计 name 里含 'reformat' 的层的耗时占比，应 < 5%
#
# 门禁 4（用分区式集成时必加）· 子图数必须 == 1
#   ORT:            ORT_TENSORRT_DUMP_SUBGRAPHS=1  + 看日志里的 subgraph 数
#   torch-tensorrt: debug=True 打印 partitioning 报告，看 unsupported ops 列表
#   # 子图 > 1 = 有算子在走慢路径 = 「转了 TRT 但没变快」的头号原因

# ══════════════════════════════════════════════════════════════════════
# C. 构建可复现性（功能安全要求「同输入同产物」，而 auto-tuning 天生不是）
# ══════════════════════════════════════════════════════════════════════
#  1. 构建机独占 + 锁频：  nvidia-smi -lgc <freq>   / jetson_clocks
#  2. --avgTiming=8        提高每个 tactic 的计时重复次数
#  3. **golden timing cache 纳入版本管理**，所有构建（CI 与发布）必须带上它
#  4. engine 旁边存一份 manifest（缺一不可，出问题要能回溯）：
#     { onnx_sha256, weights_sha256, builder_flags, workspace_mb,
#       min/opt/max_shapes, trt_version, cuda_version, driver_version,
#       gpu_name, sm_arch, timing_cache_sha256, build_host, build_time }

# ══════════════════════════════════════════════════════════════════════
# D. TSR 专项检查
# ══════════════════════════════════════════════════════════════════════
#  · 主检测器一律**静态 shape**；动态只留给两级架构第二级的 crop 分类器
#  · 第二级用**分桶 padding 到固定 batch**，别用跨度很大的单 profile
#  · RT-DETR 系：确认 grid_sample / MultiscaleDeformableAttn 是否需要 plugin
#    （RT-DETRv2 的离散采样版本可绕开；plugin 会永久切断融合链）
#  · NMS 是否用 EfficientNMS_TRT 塞进 engine：省一次 D2H + CPU 后处理，
#    但把 NMS 语义冻进了 engine，改阈值语义要重建（见模块 04）
#  · engine 矩阵：平台 x 精度 x 分辨率 x 模型版本，**每一个都是独立发布物**，
#    都要单独过精度门禁与延迟门禁。「一份权重」省的是训练，不是验证。
'''
print(RECIPE)
for token in ['--optShapes', 'timingCacheFile', 'builderOptimizationLevel',
              'exportLayerInfo', 'precisionConstraints=obey', '子图数必须 == 1',
              'onnx_sha256', 'EfficientNMS_TRT', 'grid_sample', '静态 shape']:
    assert token in RECIPE, token
print('✅ 覆盖：构建命令 / 动态 shape 三档 / 四条 CI 门禁 / 可复现性 / TSR 专项')

### 小结

- **ONNX 节点 ≠ engine 层。** 本 notebook 里 18 个 ONNX 节点融成 6 个 engine 层，数值完全一致。
  拿 ONNX 的节点数或 FLOPs 去猜 engine 性能，方向就是错的——去看 `--dumpLayerInfo`。
- **Conv+BN 融合是严格恒等**（$\tilde W=\gamma W/\sqrt{\sigma^2+\epsilon}$，$\tilde b=\gamma(b-\mu)/\sqrt{\sigma^2+\epsilon}+\beta$），
  对分组/depthwise 卷积同样成立。同一套代数还支撑 RepVGG 重参数化与 QAT 的 BN folding。
- **融合省的是访存不是计算**：标准卷积上只值 ~9%，**depthwise 上值 ~67%**。
  越是为「FLOPs 少」设计的轻量骨干，越 memory-bound，越依赖融合。
- **两条安全性条件**：中间张量消费者数 == 1，且不是图的输出。
  破坏任一条，融合安静地失败——**数值全对、评测全过、只有延迟涨了**（depthwise 骨干上实测 +33%）。
  所以你需要一条独立于一致性检查的 engine 结构门禁。
- **动态 shape 的三个数各管一件事**：`opt` 决定多快，`max` 决定占多少显存，`min` 只决定会不会崩。
  `opt` 应取「加权代价最小」而非「最常见」的 shape；多 profile 的 k=2 就吃掉 64% 的收益，k≥4 是浪费；
  **分桶不是无脑更优——粗桶的 padding 浪费会反超 profile 惩罚**。
- **engine 是 (硬件, TRT 版本, 驱动, builder config, timing cache) 的函数，还带 3–5% 的构建噪声。**
  4 平台 × 2 精度 × 3 分辨率 × 3 在网版本 = 72 个 engine，**每一个都是独立发布物**。
  最危险的不是「加载失败」（显式），而是「能加载但 tactic 不最优」（静默变慢）。

下一站：**模块 03 · INT8 校准与精度恢复** —— 本课技术密度最高的一节，也是 JD 里
「quantization accuracy drop」的直接对应。